## **Introduction to Custom Middleware**

LangChain’s modern middleware system is essentially an **"Event Hook" architecture** that allows you to tap into the lifecycle of an agent's reasoning process.

Middleware gives you all this **without rewriting the agent logic**.

Middleware exposes hooks before and after each of those steps.

<div style="display:flex; gap:20px;">
    <img src="assets/agent_loop.png" style="width:40%; height:auto;">
    <img src="assets/agent_loop_with_middlewares.png" style="width:40%; height:auto;">
</div>

### **The Execution Lifecycle: Understanding the Hooks**
In order to understand these, you must categorize them by when they fire and what they control.

1. **The "Wrapper" Middleware (The Interceptors):** They allow you to manipulate the request/response cycle directly.
    - `wrap_model_call`: Intercepts the call to the LLM. Best for modifying the ModelRequest (e.g., adding dynamic system prompts, filtering tools, forcing a specific model temperature, or implementing "self-correction" after an error).
    - `wrap_tool_call`: Intercepts the call to a tool. Best for enforcing input validation, logging tool performance, injecting authentication headers, or masking PII in tool arguments before they hit an external API.
2. **The "Lifecycle" Hooks (The Observers):** These are simpler hooks meant for side effects and logging. They do not allow you to easily "change the course" of the request like the wrappers do.
    - `before_model` / `after_model`: Triggered immediately before/after the LLM generates a response. Usage: Logging token usage, measuring latency, or capturing the raw model output for evaluation.
    - `before_agent` / `after_agent`: Triggered at the beginning/end of the entire agent loop. Usage: Setting up environment variables, cleaning up temporary resources, or performing global post-run analysis.

### **Decorator Based Middlewares**
Quick and simple for single-hook middleware. Use decorators to wrap individual functions.

**Available decorators:** 
1. Node-style
    - @before_agent - Runs before agent starts (once per invocation)
    - @before_model - Runs before each model call
    - @after_model - Runs after each model response
    - @after_agent - Runs after agent completes (once per invocation)
2. Wrap-style
    - @wrap_model_call - Wraps each model call with custom logic
    - @wrap_tool_call - Wraps each tool call with custom logic
3. Convenience
    - @dynamic_prompt - Generates dynamic system prompts

**Important Note**: `@before_model` middleware typically receives: `(state, runtime)`

In [2]:
from langchain_groq import ChatGroq

# Setup API Key
f = open('keys/.groq_api_key.txt')
GROQ_API_KEY = f.read()

chat_model = ChatGroq(
    api_key=GROQ_API_KEY, 
    model="llama-3.1-8b-instant", 
    temperature=1
)

In [1]:
from langchain.agents.middleware import before_model, AgentState
from langgraph.runtime import Runtime
from typing import Any
from langchain.messages import AIMessage

## NOTE: @before_model middleware hook should receive (state, runtime)
## You can skip to pass runtime by using (state, _)
@before_model(can_jump_to=["end"])
def check_message_limit(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    # print(runtime) 
    if len(state["messages"]) >= 5:
        return {
            "messages": [AIMessage("Conversation limit reached.")],
            "jump_to": "end"               # This helps with flow control in LangGraph
        }
    return None

In [3]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=chat_model,
    middleware=[
        check_message_limit
    ],
    checkpointer=InMemorySaver(),
)

In [4]:
response = agent.invoke(
    {"messages": "hi"},
    {"configurable" : {"thread_id" : "1"}}
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

hi
================================== Ai Message ==================================

How can I assist you today?


In [5]:
response = agent.invoke(
    {"messages": "can you tell me about langchain in one line?"},
    {"configurable" : {"thread_id" : "1"}}
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

hi
================================== Ai Message ==================================

How can I assist you today?
================================ Human Message =================================

can you tell me about langchain in one line?
================================== Ai Message ==================================

LangChain is an open-source library for building conversational AI applications, enabling developers to create custom models and integrations with various technologies, including LLMs (Large Language Models).


In [6]:
response = agent.invoke(
    {"messages": "can I use langchain to build agents?"},
    {"configurable" : {"thread_id" : "1"}}
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

hi
================================== Ai Message ==================================

How can I assist you today?
================================ Human Message =================================

can you tell me about langchain in one line?
================================== Ai Message ==================================

LangChain is an open-source library for building conversational AI applications, enabling developers to create custom models and integrations with various technologies, including LLMs (Large Language Models).
================================ Human Message =================================

can I use langchain to build agents?
================================== Ai Message ==================================

Conversation limit reached.
